In [1]:
import torch
import pandas as pd
import plotly.express as px

import numpy as np
import plotly.io as pio
from IPython.display import display, HTML

# good default for classic notebook / many Jupyter setups
pio.renderers.default = "iframe"

import sys      
sys.path.insert(0, "/storage/project/r-aivanova7-0/shared/eyas/geometry_of_truth_replication")
from src.activations import load_acts
from src.pca import run_pca

/storage/project/r-aivanova7-0/eayesh3/conda_envs/EM_env/lib/python3.14/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [40]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL   = "gemma-2-27b-it"
LAYER   = 15
DATASET_NAME = "cities"
ACTS_DIR     = "/storage/home/hcoda1/7/eayesh3/scratch/geometry_of_truth/acts"
DATASET_CSV = f"../datasets/{DATASET_NAME}.csv"

In [41]:
# load activations
city_acts = load_acts(model_name = MODEL,
                            dataset_name=DATASET_NAME,
                             layer=LAYER,
                             output_dir=ACTS_DIR)
neg_city_acts = load_acts(model_name = MODEL,
                            dataset_name="neg_cities",
                             layer=LAYER,
                             output_dir=ACTS_DIR)
larger_than_acts = load_acts(model_name = MODEL,
                            dataset_name="larger_than",
                             layer=LAYER,
                             output_dir=ACTS_DIR)
                             

In [42]:
joint_city_acts = np.concat((city_acts,neg_city_acts, larger_than_acts),axis=0)

In [43]:
joint_city_pca = run_pca(torch.from_numpy(joint_city_acts),k=50)

In [44]:
proj = joint_city_pca.projections.numpy()   # [n_statements, n_pcs]
ev   = joint_city_pca.explained_var_ratio

print(f"Projections shape: {proj.shape}")
print(f"PC1: {ev[0]:.2%}  PC2: {ev[1]:.2%}  PC3: {ev[2]:.2%}")

cumsum = np.cumsum(ev)
print(f"{np.argmax(cumsum > 0.95)+1} components until 95% var explained")

Projections shape: (4972, 50)
PC1: 40.57%  PC2: 14.22%  PC3: 5.54%
1 components until 95% var explained


In [45]:
# ── Load dataset ──────────────────────────────────────────────────────────────
df_cities = pd.read_csv(f"../datasets/cities.csv")
df_larger_than = pd.read_csv(f"../datasets/larger_than.csv")
df = pd.concat([df_cities, df_cities, df_larger_than],ignore_index=True, join="outer") 
assert len(df) == proj.shape[0], f"Row count mismatch: {len(df)} vs {proj.shape[0]}"

df["PC1"]   = proj[:, 0]
df["PC2"]   = proj[:, 1]
df["PC3"]   = proj[:, 2]
df["PC4"]   = proj[:, 3]
df["PC5"]   = proj[:, 4]
df["PC6"]   = proj[:, 5]
df["PC7"]   = proj[:, 6]
df["PC8"]   = proj[:, 7]
df["PC9"]   = proj[:, 8]
df["PC10"]   = proj[:, 9]
df["truth"] = df["label"].map({1: "True", 0: "False"})

In [46]:
# ── 2D scatter ────────────────────────────────────────────────────────────────
fig = px.scatter(
    df,
    x="PC1", y="PC2",
    color="truth",
    color_discrete_map={"True": "#d62728", "False": "#1f77b4"},
    hover_data={"statement": True, "PC1": False, "PC2": False},
    title=f"{MODEL} — cities & Larget than — layer {LAYER}  |  PC1: {ev[0]:.1%}, PC2: {ev[1]:.1%}",
    template="plotly_white",
    opacity=0.7,
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_traces(marker_size=5)
fig.show()

In [47]:
# ── 3D scatter ────────────────────────────────────────────────────────────────
fig3d = px.scatter_3d(
    df,
    x="PC1", y="PC2", z="PC3",
    color="truth",
    color_discrete_map={"True": "#d62728", "False": "#1f77b4"},
    hover_data={"statement": True, "PC1": False, "PC2": False},
    title=f"{MODEL} — Mixed Data — layer {LAYER}  |  PC1: {ev[0]:.1%}, PC2: {ev[1]:.1%}, PC3: {ev[2]:.1%}",
    template="plotly_white",
    opacity=0.7,
)
fig3d.update_traces(marker_size=3)
fig3d.show()

In [13]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-13b-hf")
print(tokenizer.padding_side)  # expected: "right"                                  



right


In [ ]:
             
from src.models import load_model
model, _ = load_model("llama-2-13b")                                                
print(model.tokenizer.padding_side)  # should be "left" — set explicitly in         


Loading LLaMA-2-13B from HuggingFace...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Wrapping with TransformerLens...
